In [2]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.utils.parallel')
import pandas as pd
import numpy as np
import heapq
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_validate, StratifiedKFold
import warnings
import copy
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. EXPANDING MEDIAN CLASS (WITH TWO HEAPS)
# ------------------------------------------------------------

class ExpandingMedian:
    """Class for calculating expanding median using two heaps"""
    def __init__(self):
        self.min_heap = []  # right half (larger values)
        self.max_heap = []  # left half (smaller values, stored as negatives)
        self.n = 0
        
    def add(self, x):
        """Add a new value and update heaps"""
        self.n += 1
        
        # Add to appropriate heap
        if not self.max_heap or x <= -self.max_heap[0]:
            heapq.heappush(self.max_heap, -x)
        else:
            heapq.heappush(self.min_heap, x)
        
        # Balance heaps
        if len(self.max_heap) > len(self.min_heap) + 1:
            heapq.heappush(self.min_heap, -heapq.heappop(self.max_heap))
        elif len(self.min_heap) > len(self.max_heap):
            heapq.heappush(self.max_heap, -heapq.heappop(self.min_heap))
    
    def get_median(self):
        """Get current median"""
        if self.n == 0:
            return 0.0
        if self.n % 2 == 1:
            return -self.max_heap[0]
        else:
            return (-self.max_heap[0] + self.min_heap[0]) / 2.0

# ------------------------------------------------------------
# 2. TREND SCORE CALCULATION FUNCTIONS (ONLY FOR NUMERIC FEATURES)
# ------------------------------------------------------------

def normalized_global_slope(window):
    """Calculate normalized global slope between -1 and 1"""
    if len(window) < 2:
        return 0
    x = np.arange(len(window))
    slope = np.polyfit(x, window, 1)[0]
    max_possible_slope = (np.max(window) - np.min(window)) / len(window) if len(window) > 1 else 1
    if max_possible_slope == 0:
        return 0
    return slope / max_possible_slope

def pairwise_slopes(window):
    """Calculate slopes between all point pairs"""
    n = len(window)
    slopes = []
    for i in range(n):
        for j in range(i+1, n):
            if j > i:
                slope = (window[j] - window[i]) / (j - i) if (j - i) != 0 else 0
                slopes.append(slope)
    return np.array(slopes)

def consistency_factor(P):
    """Consistency factor based on slope variance"""
    if len(P) == 0:
        return 0
    abs_slopes = np.abs(P)
    if np.mean(abs_slopes) == 0:
        return 0
    cv = np.std(abs_slopes) / np.mean(abs_slopes)
    return 1 / (1 + cv)

def contradiction_factor(G, P):
    """Contradiction factor between global and pairwise slopes"""
    if len(P) == 0:
        return 0
    if G >= 0:
        contradictions = np.sum(P < 0)
    else:
        contradictions = np.sum(P > 0)
    return contradictions / len(P)

def calculate_trend_score(window):
    """Calculate final trend score"""
    if len(window) < 2:  # Minimum 2 points needed
        return 0.0
        
    G = normalized_global_slope(window)
    P = pairwise_slopes(window)
    CF = consistency_factor(P)
    CD = contradiction_factor(G, P)
    
    G = np.clip(G, -1, 1)
    CF = np.clip(CF, 0, 1)
    CD = np.clip(CD, 0, 1)
    
    Trend_Score = G * CF * (1 - CD)
    return np.clip(Trend_Score, -1, 1)

# ------------------------------------------------------------
# 3. DATA PREPROCESSING FUNCTIONS
# ------------------------------------------------------------

def preprocess_data(df, target_column='label'):
    """
    Preprocess data: separate numeric and categorical features,
    encode categorical features and labels
    """
    df_processed = df.copy()
    
    # Handle target column encoding if it's categorical
    label_encoder = None
    if df_processed[target_column].dtype == 'object' or df_processed[target_column].dtype.name == 'category':
        print(f"  Encoding target column '{target_column}' from categorical to numeric...")
        label_encoder = LabelEncoder()
        df_processed[target_column] = label_encoder.fit_transform(df_processed[target_column])
        print(f"  Unique labels after encoding: {sorted(df_processed[target_column].unique())}")
    
    # Separate numeric and categorical features
    numeric_features = []
    categorical_features = []
    
    for col in df_processed.columns:
        if col == target_column:
            continue
        
        if pd.api.types.is_numeric_dtype(df_processed[col]):
            numeric_features.append(col)
        else:
            categorical_features.append(col)
    
    print(f"\nFeature types:")
    print(f"  Numeric features: {len(numeric_features)}")
    print(f"  Categorical features: {len(categorical_features)}")
    
    # Encode categorical features (simple label encoding for now)
    categorical_encoders = {}
    if categorical_features:
        print(f"\nEncoding categorical features...")
        for col in categorical_features:
            le = LabelEncoder()
            df_processed[col] = le.fit_transform(df_processed[col].astype(str))
            categorical_encoders[col] = le
            print(f"  Encoded '{col}' - {len(le.classes_)} unique values")
    
    return df_processed, numeric_features, categorical_features, categorical_encoders, label_encoder

# ------------------------------------------------------------
# 4. TREND SCORE COMPUTATION WITH PROPER WINDOW HANDLING (ONLY NUMERIC)
# ------------------------------------------------------------

def compute_incremental_trend_scores(df, feature_columns, window_size=10):
    """
    Proper window_size handling
    For first window_size-1 samples, use all available data
    NOTE: Only works with numeric features
    """
    if not feature_columns:
        # Return empty dataframe if no numeric features
        return pd.DataFrame(index=df.index)
    
    n_samples = len(df)
    n_features = len(feature_columns)
    
    # Initialize output matrix
    trend_scores = np.zeros((n_samples, n_features))
    
    for f_idx, feature in enumerate(feature_columns):
        data = df[feature].values.astype(float)
        
        for i in range(n_samples):
            if i < window_size - 1:
                # Not enough data for full window - use all available data
                window = data[:i+1]
                if len(window) >= 2:  # Need at least 2 points
                    score = calculate_trend_score(window)
                    trend_scores[i, f_idx] = score
            else:
                # Have enough data - use exactly window_size samples
                window = data[i-window_size+1:i+1]
                score = calculate_trend_score(window)
                trend_scores[i, f_idx] = score
    
    # Create column names
    columns = [f'{feature}_trend' for feature in feature_columns]
    
    return pd.DataFrame(trend_scores, columns=columns, index=df.index)

# ------------------------------------------------------------
# 5. FEATURE SELECTION WITH PROPER WINDOW HANDLING (ONLY NUMERIC)
# ------------------------------------------------------------

def select_top_trend_features(df, feature_columns, window_size=10, top_n=3):
    """
    Calculate trend scores and select top features from NUMERIC features only
    Only consider samples with full windows for strength calculation
    """
    if not feature_columns:
        print(f"\n[FEATURE SELECTION] No numeric features found! Using all features for clustering.")
        return [], pd.DataFrame(index=df.index)
    
    print(f"\n[FEATURE SELECTION] Analyzing {len(feature_columns)} numeric features...")
    print(f"  Window size: {window_size}")
    
    # Calculate trend scores
    trend_scores_df = compute_incremental_trend_scores(df, feature_columns, window_size=window_size)
    
    if trend_scores_df.empty:
        return [], trend_scores_df
    
    # Calculate trend strength for each feature
    trend_strength = {}
    for feature in feature_columns:
        trend_col = f'{feature}_trend'
        if trend_col in trend_scores_df.columns:
            # Only use samples from window_size-1 onward (where we have full windows)
            if len(trend_scores_df) >= window_size:
                valid_scores = trend_scores_df[trend_col].iloc[window_size-1:]
                strength = np.abs(valid_scores).mean()
            else:
                strength = np.abs(trend_scores_df[trend_col]).mean()
            trend_strength[feature] = strength
    
    # Sort descending and select top_n features
    sorted_features = sorted(trend_strength.items(), key=lambda x: x[1], reverse=True)
    
    # If no features have trend strength, return empty
    if not sorted_features:
        return [], trend_scores_df
    
    top_features = [f[0] for f in sorted_features[:min(top_n, len(sorted_features))]]
    
    print(f"\n   Trend strength scores (higher = stronger trend):")
    for feature, strength in sorted_features:
        marker = "✓ SELECTED" if feature in top_features else "  "
        print(f"   {marker} {feature}: {strength:.4f}")
    
    print(f"\n   Selected top {len(top_features)} features for clustering: {top_features}")
    
    # Return trend scores only for selected features
    selected_trend_cols = [f'{f}_trend' for f in top_features]
    
    return top_features, trend_scores_df[selected_trend_cols]

# ------------------------------------------------------------
# 6. INCREMENTAL STATISTICS FUNCTIONS FOR TREND SCORES
# ------------------------------------------------------------

def compute_incremental_stats_for_trend_scores(df_trend_scores):
    """
    Compute incremental statistics (mean, median, std) for trend scores
    States start fresh for each dataset
    """
    if df_trend_scores.empty or len(df_trend_scores.columns) == 0:
        return pd.DataFrame(index=df_trend_scores.index)
    
    n_samples = len(df_trend_scores)
    n_features = len(df_trend_scores.columns)
    
    # Initialize output matrix
    incremental_stats = np.zeros((n_samples, n_features * 3))
    
    # Initialize fresh states for each trend feature
    states = []
    for _ in range(n_features):
        states.append({
            'n': 0,
            'mean': 0.0,
            'M2': 0.0,  # For variance calculation (Welford's algorithm)
            'median_calculator': ExpandingMedian()
        })
    
    # Process each sample sequentially
    for i in range(n_samples):
        for f_idx, feature in enumerate(df_trend_scores.columns):
            # Get current trend score value
            x = df_trend_scores.iloc[i][feature]
            state = states[f_idx]
            
            # Update count
            state['n'] += 1
            n = state['n']
            
            # Update expanding mean (recursive formula)
            delta = x - state['mean']
            state['mean'] += delta / n
            
            # Update expanding variance (Welford's algorithm)
            delta2 = x - state['mean']
            state['M2'] += delta * delta2
            
            # Update expanding median
            state['median_calculator'].add(x)
            median_val = state['median_calculator'].get_median()
            
            # Calculate expanding std
            if n > 1:
                variance = state['M2'] / (n - 1)
                std_val = np.sqrt(variance)
            else:
                std_val = 0.0
            
            # Store results
            col_idx = f_idx * 3
            incremental_stats[i, col_idx] = state['mean']
            incremental_stats[i, col_idx + 1] = median_val
            incremental_stats[i, col_idx + 2] = std_val
    
    # Create column names
    columns = []
    for feature in df_trend_scores.columns:
        base_name = feature.replace('_trend', '')
        columns.extend([f'{base_name}_trend_mean', f'{base_name}_trend_median', f'{base_name}_trend_std'])
    
    return pd.DataFrame(incremental_stats, columns=columns, index=df_trend_scores.index)

# ------------------------------------------------------------
# 7. UNIVERSAL EDGE-PRIORITY BALANCED CLUSTERING
# ------------------------------------------------------------

def ensure_label_coverage_in_clusters(clusters, labels, X_features=None, min_threshold=10, target_count=1000):
    """
    FORCE each cluster to have at least target_count of each label
    Copy samples from clusters that have enough samples
    """
    np.random.seed(42)
    clusters = np.array(clusters)
    labels = np.array(labels)
    
    n_clusters = len(np.unique(clusters))
    unique_labels = np.unique(labels)
    
    # Create a copy of clusters to modify
    modified_clusters = clusters.copy()
    
    print(f"\n" + "="*60)
    print("FORCED LABEL COVERAGE ENHANCEMENT")
    print("="*60)
    print(f"Target: Each cluster must have at least {target_count} samples of each label")
    
    # Count initial distribution
    initial_dist = np.zeros((n_clusters, len(unique_labels)), dtype=int)
    for c in range(n_clusters):
        for l_idx, l_val in enumerate(unique_labels):
            initial_dist[c, l_idx] = np.sum((modified_clusters == c) & (labels == l_val))
    
    print("\nINITIAL DISTRIBUTION:")
    for c in range(n_clusters):
        print(f"  Cluster {c}:")
        for l_idx, l_val in enumerate(unique_labels):
            print(f"    {l_val}: {initial_dist[c, l_idx]}")
    
    # Track copied samples
    copied_info = {
        'total_copied': 0,
        'details': []
    }
    
    # Process each target cluster
    for target_cluster in range(n_clusters):
        print(f"\n" + "-"*40)
        print(f"Processing Cluster {target_cluster}")
        print("-"*40)
        
        # Process each label
        for label_idx, label_val in enumerate(unique_labels):
            current_count = np.sum((modified_clusters == target_cluster) & (labels == label_val))
            
            if current_count < target_count:
                needed = target_count - current_count
                print(f"\n  Label {label_val}: has {current_count}, needs {needed}")
                
                # Find source clusters with this label
                source_candidates = []
                for source_cluster in range(n_clusters):
                    if source_cluster != target_cluster:
                        source_count = np.sum((modified_clusters == source_cluster) & (labels == label_val))
                        if source_count > needed:  # Source has enough to share
                            source_candidates.append((source_cluster, source_count))
                
                if not source_candidates:
                    # If no cluster has enough, take from any cluster that has this label
                    for source_cluster in range(n_clusters):
                        if source_cluster != target_cluster:
                            source_count = np.sum((modified_clusters == source_cluster) & (labels == label_val))
                            if source_count > 0:
                                source_candidates.append((source_cluster, source_count))
                
                if source_candidates:
                    # Sort by distance if X_features provided
                    if X_features is not None:
                        # Calculate cluster centers
                        centers = {}
                        for c in range(n_clusters):
                            points = X_features[modified_clusters == c]
                            if len(points) > 0:
                                centers[c] = np.mean(points, axis=0)
                        
                        # Sort by distance to target cluster
                        if target_cluster in centers:
                            target_center = centers[target_cluster]
                            source_candidates.sort(key=lambda x: np.linalg.norm(target_center - centers.get(x[0], target_center)))
                    
                    # Take samples from source clusters
                    total_taken = 0
                    for source_cluster, source_count in source_candidates:
                        if total_taken >= needed:
                            break
                        
                        # How many to take from this source
                        still_needed = needed - total_taken
                        take_from_this = min(still_needed, source_count // 2)  # Take max half of source's samples
                        
                        if take_from_this > 0:
                            # Find indices of this label in source cluster
                            source_indices = np.where((modified_clusters == source_cluster) & (labels == label_val))[0]
                            
                            # Select samples to copy
                            n_to_take = min(take_from_this, len(source_indices))
                            selected = np.random.choice(source_indices, n_to_take, replace=False)
                            
                            # Copy them to target cluster
                            modified_clusters[selected] = target_cluster
                            
                            total_taken += n_to_take
                            copied_info['total_copied'] += n_to_take
                            copied_info['details'].append({
                                'from': source_cluster,
                                'to': target_cluster,
                                'label': label_val,
                                'count': n_to_take
                            })
                            
                            print(f"    Copied {n_to_take} from cluster {source_cluster}")
                    
                    if total_taken > 0:
                        print(f"    Total copied for {label_val}: {total_taken}")
                    else:
                        print(f"    WARNING: Could not copy any samples for {label_val}")
                else:
                    print(f"    ERROR: No source cluster has label {label_val}")
    
    # Final distribution
    final_dist = np.zeros((n_clusters, len(unique_labels)), dtype=int)
    for c in range(n_clusters):
        for l_idx, l_val in enumerate(unique_labels):
            final_dist[c, l_idx] = np.sum((modified_clusters == c) & (labels == l_val))
    
    print("\n" + "="*60)
    print("FINAL DISTRIBUTION")
    print("="*60)
    for c in range(n_clusters):
        print(f"\nCluster {c}:")
        for l_idx, l_val in enumerate(unique_labels):
            status = "✅" if final_dist[c, l_idx] >= target_count else f"⚠ ({final_dist[c, l_idx]}/{target_count})"
            print(f"  {l_val}: {final_dist[c, l_idx]} {status}")
    
    print(f"\nTotal samples copied: {copied_info['total_copied']}")
    
    return modified_clusters, copied_info

def balanced_cluster_assignment_with_edge_priority(X, y, kmeans_model, 
                                                   edge_threshold=0.15, 
                                                   alpha=0.3, beta=0.7,
                                                   top_k_clusters=3,
                                                   ensure_label_coverage=True,
                                                   min_threshold=10,
                                                   target_count=1000):
    """
    Universal cluster assignment for any number of labels
    Used only during training
    """
    n_samples = X.shape[0]
    n_clusters = kmeans_model.n_clusters
    
    distances = kmeans_model.transform(X)
    initial_clusters = np.argmin(distances, axis=1)
    
    unique_labels = np.unique(y)
    n_labels = len(unique_labels)
    label_distribution = np.zeros((n_clusters, n_labels))
    
    for cluster_id in range(n_clusters):
        cluster_indices = np.where(initial_clusters == cluster_id)[0]
        if len(cluster_indices) > 0:
            cluster_labels = y[cluster_indices]
            for label_idx, label_val in enumerate(unique_labels):
                label_distribution[cluster_id, label_idx] = np.sum(cluster_labels == label_val)
    
    final_clusters = np.copy(initial_clusters)
    
    for i in range(n_samples):
        sample_distances = distances[i]
        sample_label = y[i]
        label_idx = np.where(unique_labels == sample_label)[0][0]
        
        sorted_indices = np.argsort(sample_distances)
        
        if len(sorted_indices) >= 2:
            closest_dist = sample_distances[sorted_indices[0]]
            second_dist = sample_distances[sorted_indices[1]]
            
            if closest_dist > 0:
                diff_ratio = (second_dist - closest_dist) / closest_dist
            else:
                diff_ratio = 1.0
            
            if diff_ratio < edge_threshold:
                K = min(top_k_clusters, n_clusters)
                candidate_clusters = sorted_indices[:K]
                
                scores = []
                for cluster_id in candidate_clusters:
                    norm_distance = 1 - (sample_distances[cluster_id] / 
                                       np.max(sample_distances[candidate_clusters]))
                    
                    max_label_count = np.max(label_distribution[:, label_idx])
                    if max_label_count > 0:
                        label_needs = 1 - (label_distribution[cluster_id, label_idx] / max_label_count)
                    else:
                        label_needs = 1
                    
                    score = alpha * norm_distance + beta * label_needs
                    scores.append(score)
                
                best_idx = np.argmax(scores)
                final_clusters[i] = candidate_clusters[best_idx]
                
                label_distribution[final_clusters[i], label_idx] += 1
    
    # Ensure label coverage in all clusters
    if ensure_label_coverage and n_labels > 1:
        print("\n" + "="*60)
        print("ENFORCING LABEL COVERAGE IN CLUSTERS")
        print("="*60)
        final_clusters, copy_info = ensure_label_coverage_in_clusters(
            final_clusters, y, X, 
            min_threshold=min_threshold,
            target_count=target_count
        )
    
    return final_clusters

def analyze_cluster_balance(df_with_clusters, cluster_col='cluster', label_col='label'):
    """Universal cluster balance analysis"""
    n_clusters = df_with_clusters[cluster_col].nunique()
    unique_labels = sorted(df_with_clusters[label_col].unique())
    n_labels = len(unique_labels)
    
    print("="*60)
    print("CLUSTER BALANCE ANALYSIS")
    print("="*60)
    print(f"Number of clusters: {n_clusters}")
    print(f"Number of unique labels: {n_labels}")
    print(f"Labels: {unique_labels}")
    
    distribution_matrix = pd.DataFrame(
        index=[f'Cluster {i}' for i in range(n_clusters)],
        columns=[f'Label {str(l)}' for l in unique_labels]
    )
    
    balance_scores = []
    cluster_info = {}
    
    for cluster_id in range(n_clusters):
        cluster_data = df_with_clusters[df_with_clusters[cluster_col] == cluster_id]
        total_in_cluster = len(cluster_data)
        
        if total_in_cluster > 0:
            row_data = []
            for label in unique_labels:
                count = len(cluster_data[cluster_data[label_col] == label])
                percentage = (count / total_in_cluster) * 100
                distribution_matrix.loc[f'Cluster {cluster_id}', f'Label {str(label)}'] = f"{count} ({percentage:.1f}%)"
                row_data.append(count)
            
            proportions = np.array(row_data) / total_in_cluster
            
            if n_labels > 1:
                gini_impurity = 1 - np.sum(proportions**2)
                normalized_gini = gini_impurity / (1 - 1/n_labels)
            else:
                normalized_gini = 0
            
            balance_scores.append(normalized_gini)
            
            cluster_info[cluster_id] = {
                'total_samples': total_in_cluster,
                'balance_score': normalized_gini,
                'label_distribution': dict(zip(unique_labels, row_data))
            }
            
            print(f"\nCluster {cluster_id}:")
            print(f"  Total samples: {total_in_cluster}")
            if n_labels > 1:
                print(f"  Balance score: {normalized_gini:.3f} (1.0 = perfectly balanced)")
            for label, count, prop in zip(unique_labels, row_data, proportions):
                print(f"  Label {label}: {count} samples ({prop*100:.1f}%)")
    
    print("\n" + "="*60)
    print("DISTRIBUTION MATRIX")
    print("="*60)
    print(distribution_matrix)
    
    if balance_scores:
        avg_balance = np.mean(balance_scores)
        print(f"\nAverage balance score: {avg_balance:.3f}")
    else:
        avg_balance = 0
    
    return distribution_matrix, avg_balance, cluster_info

# ------------------------------------------------------------
# 8. CLUSTER-SPECIFIC FINE-TUNING WITH VALIDATION
# ------------------------------------------------------------

def fine_tune_cluster_model(base_model, X_cluster, y_cluster, X_val=None, y_val=None, 
                           fine_tuning_epochs=3, fine_tuning_fraction=0.3, random_state=42):
    """
    Fine-tune a copy of the base model on cluster-specific data
    Uses progressive fine-tuning to avoid overfitting
    """
    np.random.seed(random_state)
    
    # Create a deep copy of the base model
    cluster_model = copy.deepcopy(base_model)
    
    # If validation data is provided, use it for early stopping
    best_score = 0
    best_model = None
    
    # Progressive fine-tuning: gradually increase data usage
    n_samples = len(X_cluster)
    data_sizes = [int(n_samples * (i+1) * fine_tuning_fraction / fine_tuning_epochs) 
                  for i in range(fine_tuning_epochs)]
    data_sizes = [max(1, size) for size in data_sizes]  # Ensure at least 1 sample
    
    for epoch in range(fine_tuning_epochs):
        # Take progressively larger subsets of the cluster data
        subset_size = min(data_sizes[epoch], n_samples)
        
        # Randomly sample subset for this epoch
        if subset_size < n_samples:
            indices = np.random.choice(n_samples, subset_size, replace=False)
            X_subset = X_cluster.iloc[indices] if hasattr(X_cluster, 'iloc') else X_cluster[indices]
            y_subset = y_cluster.iloc[indices] if hasattr(y_cluster, 'iloc') else y_cluster[indices]
        else:
            X_subset = X_cluster
            y_subset = y_cluster
        
        # Train on subset
        cluster_model.fit(X_subset, y_subset)
        
        # Evaluate if validation data provided
        if X_val is not None and y_val is not None and len(X_val) > 0:
            y_pred = cluster_model.predict(X_val)
            try:
                current_score = f1_score(y_val, y_pred, average='weighted')
            except:
                current_score = accuracy_score(y_val, y_pred)
            
            if current_score > best_score:
                best_score = current_score
                best_model = copy.deepcopy(cluster_model)
    
    # Return best model if validation was used, otherwise return the fine-tuned model
    if X_val is not None and best_model is not None:
        return best_model
    else:
        return cluster_model

def train_base_model_with_cross_validation(X_train_all, y_train_all, n_folds=5, random_state=42):
    """
    Train a base model on all data with cross-validation
    Returns the trained model and validation scores
    """
    print("\n" + "="*60)
    print("TRAINING BASE MODEL ON ALL DATA")
    print("="*60)
    
    # Check if we have enough data for cross-validation
    unique_labels = np.unique(y_train_all)
    min_class_samples = np.min(np.bincount(y_train_all))
    n_folds_actual = min(n_folds, min_class_samples)
    
    # Initialize base model with balanced class weight
    base_model = RandomForestClassifier(
        n_estimators=100,
        random_state=random_state,
        class_weight='balanced',
        n_jobs=-1  # Use all cores
    )
    
    # Perform cross-validation if possible
    if n_folds_actual >= 2:
        cv = StratifiedKFold(n_splits=n_folds_actual, shuffle=True, random_state=random_state)
        
        cv_scores = cross_validate(
            base_model, X_train_all, y_train_all,
            cv=cv,
            scoring=['accuracy', 'f1_weighted', 'recall_weighted', 'precision_weighted'],
            return_train_score=True
        )
        
        print(f"Cross-validation results ({n_folds_actual}-fold):")
        print(f"  Train Accuracy: {cv_scores['train_accuracy'].mean():.4f} (+/- {cv_scores['train_accuracy'].std()*2:.4f})")
        print(f"  Val Accuracy: {cv_scores['test_accuracy'].mean():.4f} (+/- {cv_scores['test_accuracy'].std()*2:.4f})")
        print(f"  Val F1 Weighted: {cv_scores['test_f1_weighted'].mean():.4f}")
        print(f"  Val Recall Weighted: {cv_scores['test_recall_weighted'].mean():.4f}")
        print(f"  Val Precision Weighted: {cv_scores['test_precision_weighted'].mean():.4f}")
    else:
        print(f"Warning: Not enough samples for {n_folds}-fold CV. Min class samples: {min_class_samples}")
        cv_scores = None
    
    # Train final model on all data
    print("\nTraining final base model on all data...")
    base_model.fit(X_train_all, y_train_all)
    
    # Get feature importances
    if hasattr(X_train_all, 'columns'):
        feature_names = X_train_all.columns
    else:
        feature_names = [f'feature_{i}' for i in range(X_train_all.shape[1])]
    
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': base_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 most important features in base model:")
    for i, (_, row) in enumerate(feature_importance.head(10).iterrows()):
        print(f"  {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    return base_model, cv_scores, feature_importance

# ------------------------------------------------------------
# 9. MODIFIED MAIN PIPELINE WITH BASE MODEL + FINE-TUNING
# ------------------------------------------------------------

def universal_trend_clustering_pipeline(df, 
                                       target_column='label',
                                       n_clusters=3,
                                       top_n_features=3,
                                       window_size=10,
                                       test_size=0.2,
                                       random_state=42,
                                       external_test_df=None,
                                       preserve_order=True,
                                       ensure_label_coverage=True,
                                       min_threshold=10,
                                       target_count=1,
                                       use_fine_tuning=True,
                                       fine_tuning_epochs=3,
                                       fine_tuning_fraction=0.3):
    """
    MODIFIED: Pipeline with base model + fine-tuning approach
    Handles both numeric and categorical features
    """
    
    print("="*80)
    print("UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE")
    print("="*80)
    
    # --------------------------------------------------------
    # 1. Data Preprocessing
    # --------------------------------------------------------
    
    print(f"\n[1] DATA PREPROCESSING")
    
    # Preprocess data (encode categorical features and labels)
    df_processed, numeric_features, categorical_features, categorical_encoders, label_encoder = preprocess_data(
        df, target_column
    )
    
    if external_test_df is not None:
        # Preprocess test data similarly
        test_df_processed, _, _, _, _ = preprocess_data(external_test_df, target_column)
    else:
        test_df_processed = None
    
    all_features = numeric_features + categorical_features
    
    print(f"\n  Total features after encoding: {len(all_features)}")
    print(f"  Target column: {target_column}")
    print(f"  Total samples: {len(df_processed)}")
    
    label_info = df_processed[target_column].value_counts().sort_index()
    n_classes = len(label_info)
    print(f"  Number of classes: {n_classes}")
    print(f"  Class distribution:")
    for label, count in label_info.items():
        percentage = (count / len(df_processed)) * 100
        print(f"    Class {label}: {count} samples ({percentage:.1f}%)")
    
    # --------------------------------------------------------
    # 2. Train-Test Split (or use external test)
    # --------------------------------------------------------
    
    print(f"\n[2] DATA SPLITTING")
    
    if test_df_processed is not None:
        # Use external test set
        print(f"    Using external test set provided")
        train_df = df_processed.copy()
        test_df = test_df_processed.copy()
        
        print(f"    Training samples: {len(train_df)}")
        print(f"    Test samples: {len(test_df)}")
        
    else:
        # Internal train-test split
        if preserve_order:
            # Class-wise sequential split (preserves order AND label distribution)
            print(f"    Using CLASS-WISE sequential split (order preserved)")
            
            train_dfs = []
            test_dfs = []
            
            # Get unique labels
            unique_labels = df_processed[target_column].unique()
            print(f"    Splitting each of {len(unique_labels)} classes separately...")
            
            for label in unique_labels:
                # Get data for this class only
                class_data = df_processed[df_processed[target_column] == label].copy()
                n_class_samples = len(class_data)
                
                if n_class_samples > 0:
                    # Calculate split index for this class
                    split_idx = int(n_class_samples * (1 - test_size))
                    
                    # Split this class's data
                    class_train = class_data.iloc[:split_idx]
                    class_test = class_data.iloc[split_idx:]
                    
                    train_dfs.append(class_train)
                    test_dfs.append(class_test)
                    
                    print(f"      Class {label}: {n_class_samples} total -> "
                          f"{len(class_train)} train, {len(class_test)} test")
            
            # Combine all classes
            train_df = pd.concat(train_dfs, ignore_index=False).sort_index()
            test_df = pd.concat(test_dfs, ignore_index=False).sort_index()
            
            print(f"\n    Training samples: {len(train_df)}")
            print(f"    Test samples: {len(test_df)}")
            
        else:
            # Random split with stratification
            print(f"    Using random split with stratification")
            from sklearn.model_selection import train_test_split
            if n_classes > 1:
                train_df, test_df = train_test_split(
                    df_processed, 
                    test_size=test_size, 
                    stratify=df_processed[target_column], 
                    random_state=random_state
                )
            else:
                print("    Warning: Only one class found. Stratification disabled.")
                train_df, test_df = train_test_split(
                    df_processed, 
                    test_size=test_size, 
                    random_state=random_state
                )
            print(f"    Training samples: {len(train_df)}")
            print(f"    Test samples: {len(test_df)}")
    
    # --------------------------------------------------------
    # 3. FEATURE SELECTION (ONLY FOR NUMERIC FEATURES)
    # --------------------------------------------------------
    
    print(f"\n[3] FEATURE SELECTION BASED ON TREND STRENGTH")
    print(f"    Analyzing {len(numeric_features)} numeric features...")
    print(f"    Using window_size={window_size} for trend calculation")
    
    # Calculate trend scores for training data and select top numeric features
    if numeric_features:
        top_features, top_trend_scores_train = select_top_trend_features(
            train_df, numeric_features, window_size=window_size, top_n=top_n_features
        )
    else:
        print(f"\n    WARNING: No numeric features found for trend analysis!")
        top_features = []
        top_trend_scores_train = pd.DataFrame(index=train_df.index)
    
    # If no features selected for clustering, use all numeric features
    if not top_features and numeric_features:
        print(f"\n    No features selected based on trend. Using all numeric features for clustering.")
        top_features = numeric_features[:min(top_n_features, len(numeric_features))]
        top_trend_scores_train = compute_incremental_trend_scores(train_df, top_features, window_size=window_size)
    
    print(f"\n    Selected {len(top_features)} features for clustering:")
    for i, feat in enumerate(top_features):
        print(f"      {i+1}. {feat}")
    
    # --------------------------------------------------------
    # 4. Compute INCREMENTAL TREND SCORES
    # --------------------------------------------------------
    
    print(f"\n[4] COMPUTING INCREMENTAL TREND SCORES")
    
    if top_features:
        # Compute incremental statistics on trend scores
        incremental_stats_train = compute_incremental_stats_for_trend_scores(top_trend_scores_train)
        train_df_with_stats = pd.concat([train_df, incremental_stats_train], axis=1)
        incremental_feature_cols = [col for col in train_df_with_stats.columns 
                                   if col.endswith(('_trend_mean', '_trend_median', '_trend_std'))]
        
        print(f"    Created {len(incremental_feature_cols)} incremental trend features")
        print(f"    (3 statistics × {len(top_features)} features = {len(incremental_feature_cols)} features)")
    else:
        print(f"    No trend features created. Using categorical features for clustering.")
        train_df_with_stats = train_df.copy()
        incremental_feature_cols = []
    
    # --------------------------------------------------------
    # 5. Edge-Priority Balanced Clustering
    # --------------------------------------------------------
    
    print(f"\n[5] EDGE-PRIORITY BALANCED CLUSTERING")
    
    # Prepare features for clustering
    if incremental_feature_cols:
        # Use trend features if available
        clustering_features = incremental_feature_cols
        print(f"    Using TREND SCORE STATISTICS from top {len(top_features)} features")
    else:
        # Fallback to all features (categorical + numeric)
        clustering_features = all_features
        print(f"    No trend features available. Using ALL {len(all_features)} features for clustering")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_df_with_stats[clustering_features])
    
    # Adjust n_clusters if necessary
    n_clusters_actual = min(n_clusters, len(train_df))
    if n_clusters_actual < n_clusters:
        print(f"    Adjusting n_clusters from {n_clusters} to {n_clusters_actual} (not enough samples)")
        n_clusters = n_clusters_actual
    
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(X_train_scaled)
    
    BALANCING_PARAMS = {
        'edge_threshold': 0.15,
        'alpha': 0.3,
        'beta': 0.7,
        'top_k_clusters': 3,
        'ensure_label_coverage': ensure_label_coverage,
        'min_threshold': min_threshold,
        'target_count': target_count
    }
    
    train_clusters_balanced = balanced_cluster_assignment_with_edge_priority(
        X_train_scaled, 
        train_df[target_column].values, 
        kmeans_model=kmeans,
        **BALANCING_PARAMS
    )
    
    train_df_with_stats['cluster'] = train_clusters_balanced
    
    print("\n    Training Cluster Balance Results:")
    balance_matrix, avg_balance, cluster_info = analyze_cluster_balance(
        train_df_with_stats, 'cluster', target_column
    )
    
    train_df_final = train_df_with_stats
    
    # --------------------------------------------------------
    # 6. Process Test Data
    # --------------------------------------------------------
    
    print(f"\n[6] PROCESSING TEST DATA")
    
    if test_df is not None and len(test_df) > 0:
        if top_features:
            # Compute trend scores for test data
            trend_scores_test = compute_incremental_trend_scores(test_df, top_features, window_size=window_size)
            incremental_stats_test = compute_incremental_stats_for_trend_scores(trend_scores_test)
            test_df_with_stats = pd.concat([test_df, incremental_stats_test], axis=1)
            
            # Ensure test has same columns as train
            for col in incremental_feature_cols:
                if col not in test_df_with_stats.columns:
                    test_df_with_stats[col] = 0
            
            X_test_scaled = scaler.transform(test_df_with_stats[incremental_feature_cols] 
                                            if incremental_feature_cols 
                                            else test_df_with_stats[clustering_features])
        else:
            test_df_with_stats = test_df.copy()
            X_test_scaled = scaler.transform(test_df[clustering_features])
        
        # Predict clusters for test data
        test_clusters = kmeans.predict(X_test_scaled)
        test_df_with_stats['cluster'] = test_clusters
        test_df_final = test_df_with_stats
    else:
        print(f"    No test data available")
        test_df_final = pd.DataFrame()
    
    # --------------------------------------------------------
    # 7. TRAIN BASE MODEL ON ALL DATA
    # --------------------------------------------------------
    
    print(f"\n[7] TRAINING BASE MODEL ON ALL TRAINING DATA")
    print(f"    Using ALL {len(all_features)} features for classification")
    
    # Prepare all training data for base model
    X_train_all = train_df_final[all_features]
    y_train_all = train_df_final[target_column]
    
    # Train base model with cross-validation
    base_model, base_cv_scores, base_feature_importance = train_base_model_with_cross_validation(
        X_train_all, y_train_all, n_folds=5, random_state=random_state
    )
    
    # --------------------------------------------------------
    # 8. FINE-TUNE CLUSTER-SPECIFIC MODELS
    # --------------------------------------------------------
    
    print(f"\n[8] FINE-TUNING CLUSTER-SPECIFIC MODELS")
    
    fine_tuned_models = {}
    cluster_train_results = {}
    
    for i in range(n_clusters):
        # Get cluster-specific training data
        train_cluster_data = train_df_final[train_df_final['cluster'] == i]
        
        if len(train_cluster_data) < 5:
            print(f"\n    Cluster {i}: Insufficient training data ({len(train_cluster_data)} samples). Skipping fine-tuning.")
            continue
        
        X_train_cluster = train_cluster_data[all_features]
        y_train_cluster = train_cluster_data[target_column]
        
        # Get cluster-specific validation data (from test set if available)
        X_val_cluster = None
        y_val_cluster = None
        if test_df_final is not None and len(test_df_final) > 0:
            test_cluster_data = test_df_final[test_df_final['cluster'] == i]
            if len(test_cluster_data) > 0:
                X_val_cluster = test_cluster_data[all_features]
                y_val_cluster = test_cluster_data[target_column]
        
        print(f"\n    Fine-tuning Cluster {i}:")
        print(f"      Training samples: {len(train_cluster_data)}")
        print(f"      Validation samples: {len(test_cluster_data) if test_df_final is not None and len(test_cluster_data) > 0 else 0}")
        
        # Fine-tune the base model on this cluster's data
        if use_fine_tuning:
            fine_tuned_model = fine_tune_cluster_model(
                base_model, 
                X_train_cluster, 
                y_train_cluster,
                X_val_cluster,
                y_val_cluster,
                fine_tuning_epochs=fine_tuning_epochs,
                fine_tuning_fraction=fine_tuning_fraction,
                random_state=random_state
            )
        else:
            # If fine-tuning disabled, just use the base model
            fine_tuned_model = base_model
        
        fine_tuned_models[i] = fine_tuned_model
        
        # Evaluate on cluster's training data
        y_train_pred = fine_tuned_model.predict(X_train_cluster)
        
        cluster_train_results[i] = {
            'accuracy': accuracy_score(y_train_cluster, y_train_pred),
            'f1_weighted': f1_score(y_train_cluster, y_train_pred, average='weighted'),
            'n_samples': len(train_cluster_data),
            'label_distribution': y_train_cluster.value_counts().to_dict()
        }
        
        print(f"      Train Accuracy: {cluster_train_results[i]['accuracy']:.4f}")
        print(f"      Train F1 Weighted: {cluster_train_results[i]['f1_weighted']:.4f}")
    
    # --------------------------------------------------------
    # 9. Evaluate Models on Test Data
    # --------------------------------------------------------
    
    print(f"\n[9] EVALUATING MODELS ON TEST DATA")
    
    cluster_test_results = {}
    all_test_predictions = []
    all_test_true = []
    
    if test_df_final is not None and len(test_df_final) > 0:
        for i in range(n_clusters):
            test_cluster_data = test_df_final[test_df_final['cluster'] == i]
            
            if len(test_cluster_data) == 0:
                print(f"\n    Cluster {i}: No test data.")
                continue
            
            if i not in fine_tuned_models:
                print(f"\n    Cluster {i}: No fine-tuned model available. Using base model.")
                model_to_use = base_model
            else:
                model_to_use = fine_tuned_models[i]
            
            X_test_cluster = test_cluster_data[all_features]
            y_test_cluster = test_cluster_data[target_column]
            
            y_test_pred = model_to_use.predict(X_test_cluster)
            
            # Store predictions
            all_test_predictions.extend(y_test_pred)
            all_test_true.extend(y_test_cluster)
            
            # Calculate metrics
            cluster_test_results[i] = {
                'accuracy': accuracy_score(y_test_cluster, y_test_pred),
                'f1_weighted': f1_score(y_test_cluster, y_test_pred, average='weighted'),
                'recall_weighted': recall_score(y_test_cluster, y_test_pred, average='weighted'),
                'precision_weighted': precision_score(y_test_cluster, y_test_pred, average='weighted'),
                'n_samples': len(test_cluster_data),
                'confusion_matrix': confusion_matrix(y_test_cluster, y_test_pred)
            }
            
            print(f"\n    Cluster {i} Test Results:")
            print(f"      Samples: {len(test_cluster_data)}")
            print(f"      Accuracy: {cluster_test_results[i]['accuracy']:.4f}")
            print(f"      F1 Weighted: {cluster_test_results[i]['f1_weighted']:.4f}")
        
        # Overall test performance
        if all_test_true:
            overall_accuracy = accuracy_score(all_test_true, all_test_predictions)
            overall_f1 = f1_score(all_test_true, all_test_predictions, average='weighted')
            
            print(f"\n    OVERALL TEST PERFORMANCE:")
            print(f"      Total test samples: {len(all_test_true)}")
            print(f"      Accuracy: {overall_accuracy:.4f}")
            print(f"      F1 Weighted: {overall_f1:.4f}")
        else:
            overall_accuracy = overall_f1 = 0
    else:
        overall_accuracy = overall_f1 = 0
    
    # --------------------------------------------------------
    # 10. Return Results
    # --------------------------------------------------------
    
    print("\n" + "="*80)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("="*80)
    
    return {
        'train_df': train_df_final,
        'test_df': test_df_final,
        'base_model': base_model,
        'base_cv_scores': base_cv_scores,
        'base_feature_importance': base_feature_importance,
        'fine_tuned_models': fine_tuned_models,
        'cluster_train_results': cluster_train_results,
        'cluster_test_results': cluster_test_results,
        'top_features': top_features,
        'all_features': all_features,
        'cluster_info': cluster_info,
        'n_classes': n_classes,
        'kmeans_model': kmeans,
        'scaler': scaler,
        'incremental_feature_cols': incremental_feature_cols if incremental_feature_cols else all_features,
        'target_column': target_column,
        'label_encoder': label_encoder,
        'categorical_encoders': categorical_encoders,
        'window_size': window_size
    }

# ------------------------------------------------------------
# 10. DEPLOYMENT PREDICTION PIPELINE
# ------------------------------------------------------------

def create_deployment_pipeline(training_results):
    """
    Create a deployment pipeline from training results
    Handles categorical features appropriately
    """
    # Extract components from training results
    kmeans_model = training_results['kmeans_model']
    scaler = training_results['scaler']
    base_model = training_results['base_model']
    fine_tuned_models = training_results.get('fine_tuned_models', {})
    top_features = training_results['top_features']
    all_features = training_results['all_features']
    window_size = training_results.get('window_size', 10)
    label_encoder = training_results.get('label_encoder')
    categorical_encoders = training_results.get('categorical_encoders', {})
    incremental_feature_cols = training_results.get('incremental_feature_cols', [])
    
    print("\n" + "="*60)
    print("DEPLOYMENT PIPELINE CREATED")
    print("="*60)
    print(f"Using top {len(top_features)} numeric features for trend analysis")
    print(f"Total features for classification: {len(all_features)}")
    print(f"Using window size: {window_size}")
    
    # Initialize fresh states for deployment
    raw_states = {feature: [] for feature in top_features}
    
    # States for trend score statistics
    trend_states = []
    for _ in range(len(top_features)):
        trend_states.append({
            'n': 0,
            'mean': 0.0,
            'M2': 0.0,
            'median_calculator': ExpandingMedian()
        })
    
    def predict_new_sample(sample_dict):
        """
        Predict label for a new sample
        """
        # 1. Preprocess input (encode categorical features)
        processed_features = []
        
        for feature in all_features:
            if feature in sample_dict:
                value = sample_dict[feature]
                
                # Encode categorical features if needed
                if feature in categorical_encoders:
                    try:
                        # Try to transform the categorical value
                        value = categorical_encoders[feature].transform([str(value)])[0]
                    except:
                        # If unseen category, use -1 or most frequent
                        value = -1
                
                processed_features.append(value)
            else:
                # If feature is missing, use 0
                processed_features.append(0)
        
        # 2. Calculate trend scores for numeric features
        if top_features:
            trend_scores = []
            for f_idx, feature in enumerate(top_features):
                if feature in sample_dict:
                    x = float(sample_dict[feature])
                else:
                    x = 0.0
                
                # Update raw values
                raw_states[feature].append(x)
                
                # Calculate trend score
                if len(raw_states[feature]) < 2:
                    score = 0.0
                elif len(raw_states[feature]) < window_size:
                    score = calculate_trend_score(np.array(raw_states[feature]))
                else:
                    current_window = raw_states[feature][-window_size:]
                    score = calculate_trend_score(np.array(current_window))
                
                # Update statistics
                state = trend_states[f_idx]
                state['n'] += 1
                n = state['n']
                
                delta = score - state['mean']
                state['mean'] += delta / n
                
                delta2 = score - state['mean']
                state['M2'] += delta * delta2
                
                state['median_calculator'].add(score)
                median_val = state['median_calculator'].get_median()
                
                if n > 1:
                    variance = state['M2'] / (n - 1)
                    std_val = np.sqrt(variance)
                else:
                    std_val = 0.0
                
                trend_scores.extend([state['mean'], median_val, std_val])
            
            # Scale features
            if incremental_feature_cols:
                scaled_features = scaler.transform([trend_scores])
            else:
                scaled_features = scaler.transform([processed_features])
        else:
            scaled_features = scaler.transform([processed_features])
        
        # 3. Predict cluster
        cluster_id = kmeans_model.predict(scaled_features)[0]
        
        # 4. Choose model
        if cluster_id in fine_tuned_models:
            classifier = fine_tuned_models[cluster_id]
            model_type = "fine-tuned"
        else:
            classifier = base_model
            model_type = "base"
        
        # 5. Predict label
        prediction = classifier.predict([processed_features])[0]
        
        # 6. Decode label if needed
        if label_encoder is not None:
            try:
                prediction = label_encoder.inverse_transform([int(prediction)])[0]
            except:
                pass
        
        result = {
            'cluster_id': int(cluster_id),
            'model_used': model_type,
            'prediction': prediction
        }
        
        # Get probabilities if available
        if hasattr(classifier, 'predict_proba'):
            probabilities = classifier.predict_proba([processed_features])[0]
            result['probabilities'] = probabilities.tolist()
            result['confidence'] = float(np.max(probabilities))
        
        return result
    
    return predict_new_sample

# ------------------------------------------------------------
# 11. EXECUTION FUNCTIONS
# ------------------------------------------------------------

def run_mode_3_dataframes(train_df, test_df=None, preserve_order=True, **kwargs):
    """
    MODE 3: Use existing DataFrames
    """
    if test_df is None:
        print("="*80)
        print("MODE 3: SINGLE DATAFRAME")
        print("="*80)
        print(f"Note: Order preservation = {preserve_order}")
        if preserve_order:
            print("      Using CLASS-WISE sequential split")
        
        kwargs['preserve_order'] = preserve_order
        results = universal_trend_clustering_pipeline(df=train_df, **kwargs)
    else:
        print("="*80)
        print("MODE 3: TWO DATAFRAMES")
        print("="*80)
        print("Note: Order is preserved from input DataFrames")
        
        results = universal_trend_clustering_pipeline(
            df=train_df,
            external_test_df=test_df,
            preserve_order=True,
            **kwargs
        )
    return results

# ------------------------------------------------------------
# 12. EXAMPLE USAGE
# ------------------------------------------------------------

if __name__ == "__main__":
    # Example with your data
    print("Loading your data...")
    
    # Load your data
    df = pd.read_csv('DATASETS/ISCX-TOR/ISCX-TOR.csv')
    
    # Run pipeline
    results = run_mode_3_dataframes(
        train_df=df,
        target_column='label',
        n_clusters=3,
        top_n_features=3,
        test_size=0.2,
        window_size=3,
        preserve_order=True,
        ensure_label_coverage=True,
        min_threshold=10,
        target_count=1000,
        use_fine_tuning=True,
        fine_tuning_epochs=3,
        fine_tuning_fraction=0.3
    )
    
    # Create deployment pipeline
    predict_func = create_deployment_pipeline(results)
    
    # Test prediction
    if results['test_df'] is not None and len(results['test_df']) > 0:
        test_sample = results['test_df'].iloc[0].to_dict()
        prediction = predict_func(test_sample)
        print(f"\nTest prediction: {prediction}")

Loading your data...
MODE 3: SINGLE DATAFRAME
Note: Order preservation = True
      Using CLASS-WISE sequential split
UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE

[1] DATA PREPROCESSING
  Encoding target column 'label' from categorical to numeric...
  Unique labels after encoding: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

Feature types:
  Numeric features: 36
  Categorical features: 0

  Total features after encoding: 36
  Target column: label
  Total samples: 14414
  Number of classes: 8
  Class distribution:
    Class 0: 1035 samples (7.2%)
    Class 1: 1887 samples (13.1%)
    Class 2: 502 samples (3.5%)
    Class 3: 1665 samples (11.6%)
    Class 4: 461 samples (3.2%)
    Class 5: 2138 samples (14.8%)
    Class 6: 2202 samples (15.3%)
    Class 7: 4524 samples (31.4%)

[2] DATA SPLITTING
    Using CLASS-WISE sequential split (order preserved)
    Splitting each of 8 classes separately...
      Class 0: 1035 to

In [2]:
# ------------------------------------------------------------
# MAIN EXECUTION WITH COMMAND LINE INTERFACE SUPPORT
# ------------------------------------------------------------

    
# Example usage with your data
df = pd.read_csv('DATASETS/ISCX-TOR/ISCX-TOR.csv')

# Quick usage with your own DataFrame
# Example 1: Single DataFrame with internal split

# Run pipeline
results = run_mode_3_dataframes(
    train_df=df,
    target_column='label',
    n_clusters=3,
    top_n_features=3,
    test_size=0.2,
    window_size=3,
    preserve_order=True,
    ensure_label_coverage=True,
    min_threshold=10,
    target_count=1000,
    use_fine_tuning=True,
    fine_tuning_epochs=3,
    fine_tuning_fraction=0.3
)



MODE 3: SINGLE DATAFRAME
Note: Order preservation = True
      Using CLASS-WISE sequential split
UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE

[1] DATA PREPROCESSING
  Encoding target column 'label' from categorical to numeric...
  Unique labels after encoding: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

Feature types:
  Numeric features: 36
  Categorical features: 0

  Total features after encoding: 36
  Target column: label
  Total samples: 14414
  Number of classes: 8
  Class distribution:
    Class 0: 1035 samples (7.2%)
    Class 1: 1887 samples (13.1%)
    Class 2: 502 samples (3.5%)
    Class 3: 1665 samples (11.6%)
    Class 4: 461 samples (3.2%)
    Class 5: 2138 samples (14.8%)
    Class 6: 2202 samples (15.3%)
    Class 7: 4524 samples (31.4%)

[2] DATA SPLITTING
    Using CLASS-WISE sequential split (order preserved)
    Splitting each of 8 classes separately...
      Class 0: 1035 total -> 828 train, 207

In [7]:
# ------------------------------------------------------------
# MAIN EXECUTION WITH COMMAND LINE INTERFACE SUPPORT
# ------------------------------------------------------------

# Example 2: Two DataFrames (train and test already separated)
train_df = pd.read_csv('DATASETS/CDR-MLC/scale_0.001/Short/level_1.csv')
test_df = pd.read_csv('DATASETS/CDR-MLC/scale_0.001/Short/level_3.csv')

# Quick usage with your own DataFrame
# Example 1: Single DataFrame with internal split

# Run pipeline
results = run_mode_3_dataframes(
    train_df=train_df,
    test_df=test_df,
    target_column='label',
    n_clusters=3,
    top_n_features=3,
    test_size=0.2,
    window_size=3,
    preserve_order=True,
    ensure_label_coverage=True,
    min_threshold=10,
    target_count=1000,
    use_fine_tuning=True,
    fine_tuning_epochs=3,
    fine_tuning_fraction=0.3
)

MODE 3: TWO DATAFRAMES
Note: Order is preserved from input DataFrames
UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE

[1] DATA PREPROCESSING
  Encoding target column 'label' from categorical to numeric...
  Unique labels after encoding: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Feature types:
  Numeric features: 36
  Categorical features: 0
  Encoding target column 'label' from categorical to numeric...
  Unique labels after encoding: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Feature types:
  Numeric features: 36
  Categorical features: 0

  Total features after encoding: 36
  Target column: label
  Total samples: 68079
  Number of classes: 5
  Class distribution:
    Class 0: 39798 samples (58.5%)
    Class 1: 11422 samples (16.8%)
    Class 2: 6267 samples (9.2%)
    Class 3: 3127 samples (4.6%)
    Class 4: 7465 samples (11.0%)

[2] DATA SPLITTING
    Using external test set provided
    Training samples: 68079
    Test samples

## SYN DDOS

In [ ]:

# ------------------------------------------------------------
# MAIN EXECUTION WITH COMMAND LINE INTERFACE SUPPORT
# ------------------------------------------------------------

    
# Example usage with your data
df = pd.read_csv('DATASETS/SDN-TCP-SYN-DDOS/clean_standard_ddos.csv')

# Quick usage with your own DataFrame
# Example 1: Single DataFrame with internal split

# Run pipeline
results = run_mode_3_dataframes(
    train_df=df,
    target_column='Label',
    n_clusters=3,
    top_n_features=3,
    test_size=0.2,
    window_size=3,
    preserve_order=True,
    ensure_label_coverage=True,
    min_threshold=10,
    target_count=1000,
    use_fine_tuning=True,
    fine_tuning_epochs=3,
    fine_tuning_fraction=0.3
)



MODE 3: SINGLE DATAFRAME
Note: Order preservation = True
      Using CLASS-WISE sequential split
UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE

[1] DATA PREPROCESSING
  Encoding target column 'Label' from categorical to numeric...
  Unique labels after encoding: [np.int64(0), np.int64(1), np.int64(2)]

Feature types:
  Numeric features: 76
  Categorical features: 0

  Total features after encoding: 76
  Target column: Label
  Total samples: 6234
  Number of classes: 3
  Class distribution:
    Class 0: 1 samples (0.0%)
    Class 1: 3554 samples (57.0%)
    Class 2: 2679 samples (43.0%)

[2] DATA SPLITTING
    Using CLASS-WISE sequential split (order preserved)
    Splitting each of 3 classes separately...
      Class 2: 2679 total -> 2143 train, 536 test
      Class 1: 3554 total -> 2843 train, 711 test
      Class 0: 1 total -> 0 train, 1 test

    Training samples: 4986
    Test samples: 1248

[3] FEATURE SELECTION BASED ON TREND STRENGTH
    Analyzing 76 numeric features...
 

## SDN DDOS

In [5]:

# ------------------------------------------------------------
# MAIN EXECUTION WITH COMMAND LINE INTERFACE SUPPORT
# ------------------------------------------------------------

    
# Example usage with your data
df = pd.read_csv('DATASETS/AdDDoSDN/clean_standard_ddos.csv')

# Quick usage with your own DataFrame
# Example 1: Single DataFrame with internal split

# Run pipeline
results = run_mode_3_dataframes(
    train_df=df,
    target_column='Label_multi',
    n_clusters=3,
    top_n_features=3,
    test_size=0.2,
    window_size=3,
    preserve_order=True,
    ensure_label_coverage=True,
    min_threshold=10,
    target_count=1000,
    use_fine_tuning=True,
    fine_tuning_epochs=3,
    fine_tuning_fraction=0.3
)



MODE 3: SINGLE DATAFRAME
Note: Order preservation = True
      Using CLASS-WISE sequential split
UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE

[1] DATA PREPROCESSING
  Encoding target column 'Label_multi' from categorical to numeric...
  Unique labels after encoding: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Feature types:
  Numeric features: 77
  Categorical features: 0

  Total features after encoding: 77
  Target column: Label_multi
  Total samples: 268887
  Number of classes: 7
  Class distribution:
    Class 0: 1967 samples (0.7%)
    Class 1: 62188 samples (23.1%)
    Class 2: 10618 samples (3.9%)
    Class 3: 16028 samples (6.0%)
    Class 4: 44201 samples (16.4%)
    Class 5: 45186 samples (16.8%)
    Class 6: 88699 samples (33.0%)

[2] DATA SPLITTING
    Using CLASS-WISE sequential split (order preserved)
    Splitting each of 7 classes separately...
      Class 2: 10618 total -> 8494 train, 2124 test
      Class 3: 16

KeyboardInterrupt: 